# Databases
v.ekc

Continuing from last class — a full run-through of the SQLite workflow: create, populate, query, exchange with pandas, and clean up.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | Working with Databases — SQLite |
| 3 | Reading SQL Results into Pandas |
| 4 | Saving DataFrames to SQLite |
| 5 | 🔬 Activity — Chinook Database |
| Appendix | Quick Reference |

---
## 1. Setup

In [ ]:
# When we want to work with datbases
import sqlite3
import pandas as pd
import numpy as np

---
## 2. Working with Databases — SQLite

SQLite is a lightweight, file-based database. Python's built-in `sqlite3` library gives you a connection and cursor to execute SQL directly.

### 📋 Board Reference

| Step | Code | Notes |
|------|------|-------|
| Connect | `sqlite3.connect('file.db')` | Creates file if it doesn't exist |
| Cursor | `conn.cursor()` | Executes SQL statements |
| Execute | `cursor.execute("SQL")` | Run a single statement |
| Execute many | `cursor.executemany("SQL", list)` | Insert multiple rows |
| Commit | `conn.commit()` | Save changes |
| Close | `cursor.close(); conn.close()` | Always close when done |

In [ ]:
# establish a connection with the database (this will create one if it doesn't exist)
dbconn = sqlite3.connect('demo.db')

In [ ]:
type(dbconn)

In [ ]:
# Check to see if your database is in your working directory
!ls

In [ ]:
# Create a cursor to use to execute SQL statements
cursor = dbconn.cursor()

In [ ]:
# Check data type
type(cursor)

### Use the cursor to execute SQL statements to the database.
Note:  We always execute a commit after the statement.

In [ ]:
# Create a table in our database
cursor.execute("""
CREATE TABLE  books
(title TEXT, author TEXT, price FLOAT, year INTEGER)
""")
dbconn.commit()

In [ ]:
# Add rows to our table
cursor.execute("""
    INSERT INTO books VALUES 
    ("The Great Gatsby", "F. Scott Fitzgerald", 5.15, 1925)
""")
cursor.execute("""
    INSERT INTO books VALUES 
    ("1984", "George Orwell", 7.24, 1961)
    """)

dbconn.commit()

In [ ]:
# Add multiple rows to our table
cursor.execute("""
    INSERT INTO books VALUES
        ("Pride and Prejudice", "Jane Austen", 12.99, 1813),
        ("To Kill a Mockingbird", "Harper Lee", 15.50, 1960)
""")
dbconn.commit()

In [ ]:
# another way to add multiple rows
books_list = [
    ("The Catcher in the Rye", "J.D. Salinger", 14.99, 1951),
    ("The Lord of the Rings", "J.R.R. Tolkien", 19.95, 1954),
    ("Moby-Dick", "Herman Melville", 11.75, 1851)
]

cursor.executemany("""
    INSERT INTO books VALUES (?,?,?,?)
""", books_list)
dbconn.commit()

In [ ]:
# Query data
lstbooks = cursor.execute('''SELECT * FROM books''').fetchall()
print(lstbooks)

In [ ]:
# Turn this into a dataframe
pd.DataFrame(lstbooks)

---
## 3. Reading SQL Results into Pandas

`pd.read_sql_query()` is cleaner than fetching and converting manually — it handles column names automatically.

In [ ]:
# convert query results to a dataframe
dfbook = pd.read_sql_query("SELECT * FROM books", dbconn)
dfbook

In [ ]:
dfbook.dtypes

In [ ]:
# query columns
pd.read_sql_query('''SELECT title,author,year FROM books''', dbconn)

In [ ]:
# Query rows based on condition
pd.read_sql_query('''SELECT * FROM books WHERE price < 10''', dbconn)

In [ ]:
# Query row based on position
pd.read_sql_query('''SELECT * FROM books WHERE ROWID = 3''', dbconn)

In [ ]:
# Query multiple rows based on position
pd.read_sql_query('''SELECT * FROM books LIMIT 3 OFFSET 2''', dbconn)

In [ ]:
# Query every other row
pd.read_sql_query('''SELECT * FROM (
               SELECT *, ROW_NUMBER() OVER () AS row_num FROM books
           ) AS numbered_rows
           WHERE row_num % 2 = 1''', dbconn)

---
## 4. Saving DataFrames to SQLite

`df.to_sql()` lets you write a pandas DataFrame directly into a database table.

In [ ]:
# Suppose we have another dataframe
another_df = pd.read_csv('earthquakes.csv')  
another_df

In [ ]:
# Add this dataframe to our database
#if_exists{'fail', 'replace', 'append'}, default 'fail'
another_df.to_sql('earthquakes', con=dbconn, if_exists='replace')

In [ ]:
# Check if it is in the database
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type = 'table'", dbconn)

In [ ]:
cursor.execute('''DROP TABLE books''')
dbconn.commit()

In [ ]:
# Check if it is in the database
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type = 'table'", dbconn)

In [ ]:
cursor.execute('''DROP TABLE earthquakes''')
dbconn.commit()

In [ ]:
# Check if it is in the database
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type = 'table'", dbconn)

In [ ]:
# Close your cursor and connection when you are done
cursor.close()
dbconn.close()

---
## 5. 🔬 Activity — Chinook Database

Take a look at SQLite's Sample Database: [Chinook](http://www.sqlitetutorial.net/sqlite-sample-database/) — a music store database with tables for tracks, albums, artists, customers, and invoices.

---
### Try It 1: Explore the Chinook Database 😊

1. Check to see if `chinook.db` is in your working directory.

2. Connect to the chinook database and create a cursor.

3. Check what tables are in the database.

4. Read the `tracks` table into a pandas DataFrame.

5. Read the `albums` table into a pandas DataFrame.

6. Combine your `tracks_df` and `album_df` (inner join).

7. Close your connection and cursor.

In [ ]:
# 1. Check working directory


In [ ]:
# 2. Connect and create cursor
conn = ...
cur = ...

In [ ]:
# 3. Check what tables are in the database


In [ ]:
# 4. Read tracks table
tracks_df = ...
tracks_df.head()

In [ ]:
# 5. Read albums table
album_df = ...
album_df.head()

In [ ]:
# 6. Combine tracks and albums (inner join)


In [ ]:
# 7. Close connection and cursor


<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*The Chinook `tracks` and `albums` tables share an `AlbumId` key — a standard inner merge will pick it up automatically.*

```python
# 1.
!ls

# 2.
conn = sqlite3.connect('chinook.db')
cur = conn.cursor()

# 3.
pd.read_sql_query("SELECT * FROM sqlite_master WHERE type='table'", conn)

# 4.
tracks_df = pd.read_sql_query('SELECT * FROM tracks', conn)

# 5.
album_df = pd.read_sql_query('SELECT * FROM albums', conn)

# 6.
tracks_df.merge(album_df)

# 7.
cur.close()
conn.close()
```

</details>

---
## Appendix — SQLite + Pandas Quick Reference

### Connect and create
```python
import sqlite3
conn = sqlite3.connect('mydb.db')
cursor = conn.cursor()

cursor.execute("CREATE TABLE t (col1 TEXT, col2 INT)")
cursor.execute("INSERT INTO t VALUES ('a', 1)")
cursor.executemany("INSERT INTO t VALUES (?,?)", [('b',2), ('c',3)])
conn.commit()
```

### Query into pandas
```python
pd.read_sql_query("SELECT * FROM t", conn)
pd.read_sql_query("SELECT col1 FROM t WHERE col2 > 1", conn)
pd.read_sql_query("SELECT * FROM t LIMIT 5 OFFSET 2", conn)
```

### DataFrame → database
```python
df.to_sql('table_name', con=conn, if_exists='replace')
```

### Cleanup
```python
cursor.execute("DROP TABLE t")
conn.commit()
cursor.close()
conn.close()
```